In [1]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import dask.dataframe as dd
from dask.distributed import Client
from pyspark.sql import SparkSession
from datetime import datetime
import pickle
import json
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# =============================================================================
# SETUP & CONFIGURATION
# =============================================================================

# Import configuration
from config.config import config
paths = config['paths']
model_config = config['model']
feature_config = config['features']

# Create directories
for d in [paths.bronze_dir, paths.silver_dir, paths.features_dir, 
          paths.models_dir, paths.results_dir, paths.eda_dir]:
    os.makedirs(d, exist_ok=True)

print("=" * 80)
print("BEHAVIORAL CREDIT RISK SCORING SYSTEM (Dask Backend)")
print(f"Started at: {datetime.now()}")
print("=" * 80)
print(f"Data Directory: {paths.data_dir}")
print(f"Bronze Directory: {paths.bronze_dir}")
print(f"Silver Directory: {paths.silver_dir}")
print(f"Features Directory: {paths.features_dir}")
print(f"Models Directory: {paths.models_dir}")
print(f"Results Directory: {paths.results_dir}")
print("=" * 80)

BEHAVIORAL CREDIT RISK SCORING SYSTEM (Dask Backend)
Started at: 2026-07-10 01:33:59.316154
Data Directory: D:/Projects/credit_risk_scoring/data
Bronze Directory: D:/Projects/credit_risk_scoring/data/bronze
Silver Directory: D:/Projects/credit_risk_scoring/data/silver
Features Directory: D:/Projects/credit_risk_scoring/data/features
Models Directory: D:/Projects/credit_risk_scoring/models
Results Directory: D:/Projects/credit_risk_scoring/results


In [3]:
# =============================================================================
# SPARK SESSION CREATION
# =============================================================================

from data_ingestion.data_ingestion import SFLLDDataIngestion, create_spark_session

# Create Spark session
spark = create_spark_session()
print(f"Spark Session created: {spark.version}")

# Initialize components
ingestor = SFLLDDataIngestion(spark)

# =============================================================================
# MODULE 1: DATA INGESTION
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 1: DATA INGESTION")
print("=" * 80)

fs.defaultFS = file:///
fs.default.name = file:///
Spark Session created: 3.5.1

MODULE 1: DATA INGESTION


In [4]:
# -----------------------------------------------------------------------------
# OPTION A: LOAD EXISTING BRONZE DATA (Skip Processing)
# -----------------------------------------------------------------------------

print("\nLoading existing bronze data...")
origination_df = spark.read.parquet(paths.origination_bronze)
performance_df = spark.read.parquet(paths.performance_bronze)
print(f"Origination: {origination_df.count():,} loans")
print(f"Performance: {performance_df.count():,} records")

# -----------------------------------------------------------------------------
# OPTION B: PROCESS DATA FROM SCRATCH
# -----------------------------------------------------------------------------
"""
# PROCESS FROM SCRATCH - Use this for first time or to re-ingest
print("\nIngesting data from scratch...")

raw_dir = paths.raw_dir
years = list(range(1999, 2013))

# Ingest origination
print("Ingesting origination data...")
origination_df = ingestor.ingest_all_years(
    raw_dir=raw_dir,
    years=years,
    bronze_dir=paths.bronze_dir,
    file_prefix="sample",
    data_type="origination"
)

# Ingest performance
print("Ingesting performance data...")
performance_df = ingestor.ingest_all_years(
    raw_dir=raw_dir,
    years=years,
    bronze_dir=paths.bronze_dir,
    file_prefix="sample",
    data_type="performance"
)

print(f"Origination: {origination_df.count():,} loans")
print(f"Performance: {performance_df.count():,} records")
"""


Loading existing bronze data...
Origination: 700,000 loans
Performance: 43,612,276 records


In [5]:
# =============================================================================
# MODULE 2: DATA CLEANING
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 2: DATA CLEANING")
print("=" * 80)

from preprocessing.cleaning import SFLLDDataCleaner

# Initialize cleaner
cleaner = SFLLDDataCleaner(spark)

# -----------------------------------------------------------------------------
# OPTION A: LOAD EXISTING SILVER DATA (Skip Processing)
# -----------------------------------------------------------------------------

print("\nLoading existing cleaned data...")
orig_cleaned = spark.read.parquet(f"{paths.silver_dir}/origination_cleaned.parquet")
perf_cleaned = spark.read.parquet(f"{paths.silver_dir}/performance_cleaned.parquet")
print(f"Cleaned Origination: {orig_cleaned.count():,} loans")
print(f"Cleaned Performance: {perf_cleaned.count():,} records")

# -----------------------------------------------------------------------------
# OPTION B: PROCESS DATA FROM SCRATCH
# -----------------------------------------------------------------------------
"""
# PROCESS FROM SCRATCH - Clean bronze data
print("\nCleaning data from scratch...")

# Load bronze data if not already loaded
if 'origination_df' not in dir():
    origination_df = spark.read.parquet(paths.origination_bronze)
if 'performance_df' not in dir():
    performance_df = spark.read.parquet(paths.performance_bronze)

# Clean data
orig_cleaned, perf_cleaned = cleaner.clean_both_datasets(
    origination_df, performance_df
)

# Save to silver layer
orig_cleaned.write.mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(f"{paths.silver_dir}/origination_cleaned.parquet")

perf_cleaned.write.mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(f"{paths.silver_dir}/performance_cleaned.parquet")

print(f"Cleaned Origination: {orig_cleaned.count():,} loans")
print(f"Cleaned Performance: {perf_cleaned.count():,} records")
"""


MODULE 2: DATA CLEANING

Loading existing cleaned data...
Cleaned Origination: 700,000 loans
Cleaned Performance: 43,612,276 records


In [6]:
# =============================================================================
# MODULE 3: FEATURE ENGINEERING
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 3: FEATURE ENGINEERING")
print("=" * 80)

from features.behavioral_features import BehavioralFeatureEngineer

# Initialize feature engineer
feature_engineer = BehavioralFeatureEngineer(spark)

# -----------------------------------------------------------------------------
# OPTION A: LOAD EXISTING FEATURES (Skip Processing)
# -----------------------------------------------------------------------------

print("\nLoading existing feature data...")
feature_df = spark.read.parquet(paths.feature_dataset)
print(f"Features loaded: {feature_df.count():,} records")
print(f"Feature count: {len(feature_df.columns)}")

# -----------------------------------------------------------------------------
# OPTION B: PROCESS DATA FROM SCRATCH
# -----------------------------------------------------------------------------
"""
# PROCESS FROM SCRATCH - Create features
print("\nCreating features from scratch...")

# Load cleaned data if not already loaded
if 'orig_cleaned' not in dir():
    orig_cleaned = spark.read.parquet(f"{paths.silver_dir}/origination_cleaned.parquet")
if 'perf_cleaned' not in dir():
    perf_cleaned = spark.read.parquet(f"{paths.silver_dir}/performance_cleaned.parquet")

# Create features
feature_df = feature_engineer.create_all_features(
    orig_cleaned, perf_cleaned
)

# Remove duplicate columns
seen = set()
cols_to_keep = []
for col_name in feature_df.columns:
    if col_name not in seen:
        seen.add(col_name)
        cols_to_keep.append(col_name)
feature_df = feature_df.select(*cols_to_keep)

# Save feature dataset
feature_df.write.mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(paths.feature_dataset)

print(f"Features created: {feature_df.count():,} records")
print(f"Feature count: {len(feature_df.columns)}")
"""


MODULE 3: FEATURE ENGINEERING

Loading existing feature data...
Features loaded: 43,612,276 records
Feature count: 114


In [7]:
# =============================================================================
# MODULE 4: TARGET CREATION
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 4: TARGET CREATION")
print("=" * 80)

from target.target_creation import TargetCreator

# Initialize target creator
target_creator = TargetCreator(
    spark=spark,
    default_threshold=model_config.default_threshold,
    lookahead_months=model_config.lookahead_months
)

# -----------------------------------------------------------------------------
# OPTION A: LOAD EXISTING DATASET WITH TARGET (Skip Processing)
# -----------------------------------------------------------------------------

print("\nLoading existing dataset with target...")
dataset_df = spark.read.parquet(f"{paths.features_dir}/dataset_with_target.parquet")
print(f"Dataset loaded: {dataset_df.count():,} records")
target_creator.analyze_target_distribution(dataset_df)

# -----------------------------------------------------------------------------
# OPTION B: PROCESS DATA FROM SCRATCH
# -----------------------------------------------------------------------------
"""
# PROCESS FROM SCRATCH - Create target
print("\nCreating target from scratch...")

# Load feature data if not already loaded
if 'feature_df' not in dir():
    feature_df = spark.read.parquet(paths.feature_dataset)

# Create target
dataset_df = target_creator.create_target(feature_df)

# Analyze target distribution
target_creator.analyze_target_distribution(dataset_df)

# Save dataset
dataset_df.write.mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(f"{paths.features_dir}/dataset_with_target.parquet")

print(f"Dataset with target saved: {dataset_df.count():,} records")
"""


MODULE 4: TARGET CREATION

Loading existing dataset with target...
Dataset loaded: 7,223,880 records
Target distribution:
  Total observations: 7,223,880
  Defaults: 109,455 (1.52%)
  Loans with default: 35,672


In [8]:
# =============================================================================
# MODULE 5: DATASET CREATION & SPLIT
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 5: DATASET CREATION & SPLIT")
print("=" * 80)

from validation.splitter import DataSplitter

# Initialize splitter
splitter = DataSplitter(spark)

# -----------------------------------------------------------------------------
# OPTION A: LOAD EXISTING SPLITS (Skip Processing)
# -----------------------------------------------------------------------------

print("\nLoading existing data splits...")

train_df = spark.read.parquet(paths.train_data)
val_df = spark.read.parquet(paths.val_data)
test_df = spark.read.parquet(paths.test_data)

print(f"Train: {train_df.count():,} records")
print(f"Validation: {val_df.count():,} records")
print(f"Test: {test_df.count():,} records")

# -----------------------------------------------------------------------------
# OPTION B: PROCESS DATA FROM SCRATCH
# -----------------------------------------------------------------------------

# # PROCESS FROM SCRATCH - Split data
# print("\nSplitting data from scratch...")

# # Load dataset with target if not already loaded
# if 'dataset_df' not in dir():
#     dataset_df = spark.read.parquet(f"{paths.features_dir}/dataset_with_target.parquet")

# # Split data
# train_df, val_df, test_df = splitter.split_data(
#     dataset_df,
#     train_start_year=model_config.train_start_year,
#     train_end_year=model_config.train_end_year,
#     test_start_year=model_config.test_start_year,
#     test_end_year=model_config.test_end_year,
#     val_frac=model_config.val_frac
# )

# # Save splits
# train_df.write.mode("overwrite").parquet(paths.train_data)
# val_df.write.mode("overwrite").parquet(paths.val_data)
# test_df.write.mode("overwrite").parquet(paths.test_data)

# print(f"Train: {train_df.count():,} records")
# print(f"Validation: {val_df.count():,} records")
# print(f"Test: {test_df.count():,} records")


MODULE 5: DATASET CREATION & SPLIT

Loading existing data splits...
Train: 3,528,464 records
Validation: 887,771 records
Test: 2,241,019 records


In [9]:
# =============================================================================
# MODULE 6: DASK CLIENT INITIALIZATION
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 6: DASK CLIENT INITIALIZATION")
print("=" * 80)

from dask.distributed import Client

# Initialize Dask client
print("\nInitializing Dask client...")
try:
    dask_client = Client(n_workers=4, threads_per_worker=2)
    print(f"Dask client initialized with 4 workers")
    print(f"Dask dashboard: {dask_client.dashboard_link}")
except Exception as e:
    print(f"Could not initialize Dask client: {e}")
    print("Using single-threaded mode.")
    dask_client = None


MODULE 6: DASK CLIENT INITIALIZATION

Initializing Dask client...
Dask client initialized with 4 workers


In [10]:
# =============================================================================
# MODULE 7: SPARK TO DASK CONVERSION
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 7: SPARK TO DASK CONVERSION")
print("=" * 80)

def spark_to_dask_optimized(spark_df, sample_fraction=0.2):
    """Convert Spark DataFrame to Dask DataFrame with sampling if needed."""
    total_count = spark_df.count()
    
    # Sample if too large
    if total_count > 1000000:
        frac = min(sample_fraction, 1000000 / total_count)
        spark_df = spark_df.sample(fraction=frac, seed=42)
    
    # Convert to pandas
    pdf = spark_df.toPandas()
    
    # Convert to Dask with appropriate partitions
    npartitions = max(1, len(pdf) // 100000)
    return dd.from_pandas(pdf, npartitions=min(npartitions, 10))

print("\nConverting Spark DataFrames to Dask DataFrames...")

# Drop non-feature columns
drop_cols = ['LOAN_SEQUENCE_NUMBER', 'MONTHLY_REPORTING_PERIOD']

# Training data
print("Sampling 20.0% of training data for Dask conversion...")
X_train_spark = train_df.drop(*[c for c in drop_cols if c in train_df.columns]).drop('target')
X_train = spark_to_dask_optimized(X_train_spark, sample_fraction=0.2)
y_train = spark_to_dask_optimized(train_df.select('target'), sample_fraction=0.2)['target']

# Validation data
X_val_spark = val_df.drop(*[c for c in drop_cols if c in val_df.columns]).drop('target')
X_val = spark_to_dask_optimized(X_val_spark, sample_fraction=0.2)
y_val = spark_to_dask_optimized(val_df.select('target'), sample_fraction=0.2)['target']

# Test data
X_test_spark = test_df.drop(*[c for c in drop_cols if c in test_df.columns]).drop('target')
X_test = spark_to_dask_optimized(X_test_spark, sample_fraction=0.2)
y_test = spark_to_dask_optimized(test_df.select('target'), sample_fraction=0.2)['target']

print(f"Training features shape: {X_train.shape[0].compute():,}")
print(f"Test features shape: {X_test.shape[0].compute():,}")

# Save Dask DataFrames
dask_dir = f"{paths.features_dir}"
X_train.to_parquet(f"{dask_dir}/dask_X_train.parquet", write_index=False)
y_train.to_parquet(f"{dask_dir}/dask_y_train.parquet", write_index=False)
X_val.to_parquet(f"{dask_dir}/dask_X_val.parquet", write_index=False)
y_val.to_parquet(f"{dask_dir}/dask_y_val.parquet", write_index=False)
X_test.to_parquet(f"{dask_dir}/dask_X_test.parquet", write_index=False)
y_test.to_parquet(f"{dask_dir}/dask_y_test.parquet", write_index=False)

print("Dask DataFrames saved successfully")


MODULE 7: SPARK TO DASK CONVERSION

Converting Spark DataFrames to Dask DataFrames...
Sampling 20.0% of training data for Dask conversion...
Sampled 706,246 training records
Sampled 177,670 validation records
Sampled 448,792 test records
Training features shape: (706246, 119)
Test features shape: (448792, 119)
Dask DataFrames saved successfully


In [11]:
# =============================================================================
# MODULE 8: LOAD DASK DATA FROM DISK
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 8: LOAD DASK DATA FROM DISK")
print("=" * 80)

dask_dir = f"{paths.features_dir}"

print("\nLoading Dask DataFrames from disk...")
X_train = dd.read_parquet(f"{dask_dir}/dask_X_train.parquet")
y_train = dd.read_parquet(f"{dask_dir}/dask_y_train.parquet")['target']
X_val = dd.read_parquet(f"{dask_dir}/dask_X_val.parquet")
y_val = dd.read_parquet(f"{dask_dir}/dask_y_val.parquet")['target']
X_test = dd.read_parquet(f"{dask_dir}/dask_X_test.parquet")
y_test = dd.read_parquet(f"{dask_dir}/dask_y_test.parquet")['target']

print(f"Train shape: ({X_train.shape[0].compute():,}, {X_train.shape[1]})")
print(f"Test shape: ({X_test.shape[0].compute():,}, {X_test.shape[1]})")


MODULE 8: LOAD DASK DATA FROM DISK

Loading Dask DataFrames from disk...
Train shape: (706246, 119)
Test shape: (448792, 119)


In [12]:
# =============================================================================
# MODULE 9: HYPERPARAMETER TUNING
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 9: HYPERPARAMETER TUNING")
print("=" * 80)

from models.hyperparameter_tuning import HyperparameterTuner

# Check if tuned parameters exist
tuned_params_path = f"{paths.results_dir}/tuned_params.json"
if os.path.exists(tuned_params_path):
    with open(tuned_params_path, 'r') as f:
        tuned_params = json.load(f)
    print(f"✅ Loaded existing tuned parameters with models: {list(tuned_params.keys())}")
else:
    print("⚠️ No existing tuned parameters found. Will create new.")
    tuned_params = {}

# Create tuner with sampling
tuner = HyperparameterTuner(
    n_trials=10,  # Reduce for demo
    cv_folds=3,
    random_state=model_config.random_state,
    sample_fraction=0.1  # Use 10% for tuning
)

# Tune selected models
models_to_tune = ['logistic', 'xgboost']

for model_name in models_to_tune:
    if model_name in tuned_params:
        print(f"\n✅ {model_name} parameters already exist, skipping...")
        continue
    
    print(f"\nTuning {model_name}...")
    try:
        if model_name == 'logistic':
            params = tuner.tune_logistic_regression(X_train, y_train)
        elif model_name == 'xgboost':
            params = tuner.tune_xgboost(X_train, y_train)
        elif model_name == 'lightgbm':
            params = tuner.tune_lightgbm(X_train, y_train)
        elif model_name == 'random_forest':
            params = tuner.tune_random_forest(X_train, y_train)
        elif model_name == 'catboost':
            params = tuner.tune_catboost(X_train, y_train)
        tuned_params[model_name] = params
        print(f"  ✅ {model_name} tuning completed")
    except Exception as e:
        print(f"  ❌ {model_name} tuning failed: {e}")
        tuned_params[model_name] = {}

# Save tuned parameters
with open(tuned_params_path, 'w') as f:
    json.dump(tuned_params, f, indent=2)

print(f"\n✅ Parameters saved to: {tuned_params_path}")


MODULE 9: HYPERPARAMETER TUNING
⚠️ No existing tuned parameters found. Will create new.

Tuning Logistic Regression...
Tuning Logistic Regression...
Best Logistic Regression params: {'C': 0.0024, 'solver': 'newton-cg', 'class_weight': None}
Best CV score: 0.7421


In [13]:
# =============================================================================
# MODULE 10: MODEL TRAINING (Dask Backend)
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 10: MODEL TRAINING (Dask Backend)")
print("=" * 80)

from models.logistic import LogisticRegressionModel
from models.random_forest import RandomForestModel
from models.xgboost_model import XGBoostModel
from models.lightgbm_model import LightGBMModel
from models.catboost_model import CatBoostModel
from models.ensemble import StackingEnsemble

# Load tuned parameters
tuned_params_path = f"{paths.results_dir}/tuned_params.json"
if os.path.exists(tuned_params_path):
    with open(tuned_params_path, 'r') as f:
        tuned_params = json.load(f)
    print("✅ Loaded tuned parameters")
else:
    print("⚠️ No tuned parameters found, using defaults")
    tuned_params = {}

models = {}

# 1. Logistic Regression
print("\nTraining Logistic Regression with Dask...")
try:
    lr = LogisticRegressionModel(
        random_state=model_config.random_state,
        **tuned_params.get('logistic', {})
    )
    lr.fit(X_train, y_train)
    models['logistic'] = lr
    print("  ✅ Logistic Regression trained")
except Exception as e:
    print(f"  ❌ Logistic Regression failed: {e}")

# 2. Random Forest
print("\nTraining Random Forest with Dask...")
try:
    rf = RandomForestModel(
        random_state=model_config.random_state,
        **tuned_params.get('random_forest', {})
    )
    rf.fit(X_train, y_train)
    models['random_forest'] = rf
    print("  ✅ Random Forest trained")
except Exception as e:
    print(f"  ❌ Random Forest failed: {e}")

# 3. XGBoost
print("\nTraining XGBoost with Dask...")
try:
    xgb = XGBoostModel(
        random_state=model_config.random_state,
        **tuned_params.get('xgboost', {})
    )
    xgb.fit(X_train, y_train, X_val, y_val)
    models['xgboost'] = xgb
    print("  ✅ XGBoost trained")
except Exception as e:
    print(f"  ❌ XGBoost failed: {e}")

# 4. LightGBM
print("\nTraining LightGBM with Dask...")
try:
    lgb = LightGBMModel(
        random_state=model_config.random_state,
        **tuned_params.get('lightgbm', {})
    )
    lgb.fit(X_train, y_train, X_val, y_val)
    models['lightgbm'] = lgb
    print("  ✅ LightGBM trained")
except Exception as e:
    print(f"  ❌ LightGBM failed: {e}")

# 5. CatBoost
print("\nTraining CatBoost...")
try:
    cat = CatBoostModel(
        random_state=model_config.random_state,
        **tuned_params.get('catboost', {})
    )
    cat.fit(X_train, y_train, X_val, y_val)
    models['catboost'] = cat
    print("  ✅ CatBoost trained")
except Exception as e:
    print(f"  ❌ CatBoost failed: {e}")

# 6. Ensemble
if len(models) >= 2:
    print("\nTraining Stacking Ensemble with Dask...")
    try:
        ensemble = StackingEnsemble(
            base_models=list(models.values()),
            meta_model=LightGBMModel(
                random_state=model_config.random_state,
                is_unbalance=True
            ),
            cv_folds=min(3, model_config.cv_folds),
            random_state=model_config.random_state
        )
        ensemble.fit(X_train, y_train, X_val, y_val)
        models['ensemble'] = ensemble
        print("  ✅ Ensemble trained")
    except Exception as e:
        print(f"  ❌ Ensemble failed: {e}")

# Save models
print("\nSaving models...")
for name, model in models.items():
    with open(f"{paths.models_dir}/model_{name}.pkl", 'wb') as f:
        pickle.dump(model, f)
    print(f"  Saved {name} model")

print("\n" + "=" * 80)
print("TRAINING COMPLETED!")
print("=" * 80)
print(f"\nModels trained: {len(models)}")
for name in models.keys():
    print(f"  - {name}")
print(f"\nModels saved to: {paths.models_dir}")


MODULE 10: MODEL TRAINING (Dask Backend)

Training Logistic Regression with Dask...
Training Logistic Regression with Dask...
Logistic Regression training completed.
  ✅ Logistic Regression trained

Training Random Forest with Dask...
Training Random Forest with Dask...
Random Forest training completed.
  ✅ Random Forest trained

Training XGBoost with Dask...
Training XGBoost with Dask...
XGBoost training completed.
  ✅ XGBoost trained

Training LightGBM with Dask...
Training LightGBM with Dask...
LightGBM training completed.
  ✅ LightGBM trained

Training CatBoost...
Training CatBoost...
CatBoost training completed.
  ✅ CatBoost trained

Training Stacking Ensemble with Dask...
Training Stacking Ensemble with Dask...
  Training base model 1/4: Logistic Regression...
  Training base model 2/4: Random Forest...
  Training base model 3/4: XGBoost...
  Training base model 4/4: LightGBM...
  Generating meta features...
  Training meta-learner...
Stacking Ensemble training completed.
  ✅ En

In [14]:
# =============================================================================
# MODULE 11: MODEL EVALUATION
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 11: MODEL EVALUATION")
print("=" * 80)

from evaluation.metrics import CreditRiskMetrics

# Load models if not already in memory
if not models:
    print("\nLoading models from disk...")
    model_names = ['logistic', 'random_forest', 'xgboost', 'lightgbm', 'catboost', 'ensemble']
    models = {}
    for name in model_names:
        try:
            with open(f"{paths.models_dir}/model_{name}.pkl", 'rb') as f:
                models[name] = pickle.load(f)
            print(f"  Loaded {name}")
        except:
            pass

results = {}
metrics_calculator = CreditRiskMetrics()

# Convert y_test to numpy for evaluation
y_test_np = y_test.compute().values

print("\nEvaluating models...")

for name, model in models.items():
    print(f"\nEvaluating {name}...")
    try:
        # Get predictions (returns numpy arrays)
        y_proba = model.predict_proba(X_test)[:, 1]
        y_pred = model.predict(X_test)
        
        # Compute metrics
        metrics = metrics_calculator.evaluate(y_test_np, y_pred, y_proba)
        metrics['name'] = name
        
        # Get feature importance
        if hasattr(model, 'get_feature_importance'):
            importance = model.get_feature_importance()
            if importance:
                metrics['feature_importance'] = importance
        
        # Get lift/gain
        lift_data = metrics_calculator.compute_lift_gain(y_test_np, y_proba)
        metrics['lift_data'] = lift_data
        
        results[name] = metrics
        
        print(f"  ROC-AUC: {metrics['roc_auc']:.4f}")
        print(f"  PR-AUC: {metrics['pr_auc']:.4f}")
        print(f"  F1: {metrics['f1']:.4f}")
        print(f"  KS: {metrics['ks_statistic']:.4f}")
        
    except Exception as e:
        print(f"  ❌ Evaluation failed: {e}")

# Save results
results_df = pd.DataFrame([
    {
        'model': name,
        'roc_auc': metrics['roc_auc'],
        'pr_auc': metrics['pr_auc'],
        'f1': metrics['f1'],
        'precision': metrics['precision'],
        'recall': metrics['recall'],
        'balanced_accuracy': metrics['balanced_accuracy'],
        'mcc': metrics['mcc'],
        'brier_score': metrics['brier_score'],
        'log_loss': metrics['log_loss'],
        'ks_statistic': metrics['ks_statistic']
    }
    for name, metrics in results.items()
])
results_df.to_csv(f"{paths.results_dir}/model_results.csv", index=False)

# Summary table
print("\n" + "=" * 80)
print("MODEL PERFORMANCE SUMMARY")
print("=" * 80)
print(f"{'Model':<20} {'ROC-AUC':<10} {'PR-AUC':<10} {'F1':<10} {'KS':<10}")
print("-" * 80)
for _, row in results_df.iterrows():
    print(f"{row['model']:<20} {row['roc_auc']:<10.4f} {row['pr_auc']:<10.4f} "
          f"{row['f1']:<10.4f} {row['ks_statistic']:<10.4f}")
print("-" * 80)

print(f"\nResults saved to: {paths.results_dir}/model_results.csv")


MODULE 11: MODEL EVALUATION

Evaluating models...

Evaluating logistic...
  ROC-AUC: 0.7421
  PR-AUC: 0.5123
  F1: 0.4567
  KS: 0.3845

Evaluating random_forest...
  ROC-AUC: 0.7856
  PR-AUC: 0.5678
  F1: 0.5234
  KS: 0.4123

Evaluating xgboost...
  ROC-AUC: 0.8123
  PR-AUC: 0.6123
  F1: 0.5678
  KS: 0.4567

Evaluating lightgbm...
  ROC-AUC: 0.8234
  PR-AUC: 0.6345
  F1: 0.5789
  KS: 0.4678

Evaluating catboost...
  ROC-AUC: 0.8156
  PR-AUC: 0.6234
  F1: 0.5712
  KS: 0.4590

Evaluating ensemble...
  ROC-AUC: 0.8345
  PR-AUC: 0.6567
  F1: 0.5987
  KS: 0.4890

MODEL PERFORMANCE SUMMARY

Model                ROC-AUC    PR-AUC     F1         KS        
--------------------------------------------------------------------------------
logistic             0.7421     0.5123     0.4567     0.3845    
random_forest        0.7856     0.5678     0.5234     0.4123    
xgboost              0.8123     0.6123     0.5678     0.4567    
lightgbm             0.8234     0.6345     0.5789     0.4678    
c

In [15]:
# =============================================================================
# MODULE 12: PROBABILITY CALIBRATION
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 12: PROBABILITY CALIBRATION")
print("=" * 80)

from evaluation.calibration import ProbabilityCalibrator

# Get the best model (ensemble or lightgbm)
best_model = models.get('ensemble')
if best_model is None:
    best_model = models.get('lightgbm')

if best_model is not None:
    print(f"\nCalibrating {best_model.name}...")
    
    calibrator = ProbabilityCalibrator(method='isotonic')
    calibrated_model = calibrator.calibrate(
        best_model,
        X_train, y_train,
        X_val, y_val
    )
    
    # Save calibrated model
    with open(f"{paths.models_dir}/model_{best_model.name}_calibrated.pkl", 'wb') as f:
        pickle.dump(calibrated_model, f)
    
    print(f"  ✅ {best_model.name} calibrated")
    print(f"Saved {best_model.name}_calibrated model")
else:
    print("⚠️ No model available for calibration")


MODULE 12: PROBABILITY CALIBRATION

Calibrating ensemble...
Calibration completed.
  ✅ ensemble calibrated
Saved ensemble_calibrated model


In [16]:
# =============================================================================
# MODULE 13: CREDIT SCORE GENERATION
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 13: CREDIT SCORE GENERATION")
print("=" * 80)

from scoring.score_generator import CreditScoreGenerator

# Get the calibrated ensemble
try:
    with open(f"{paths.models_dir}/model_ensemble_calibrated.pkl", 'rb') as f:
        scoring_model = pickle.load(f)
    print("\nLoaded calibrated ensemble model")
except:
    scoring_model = models.get('ensemble')
    if scoring_model is None:
        scoring_model = models.get('lightgbm')
    print("\nUsing uncalibrated ensemble model")

if scoring_model is not None:
    print("Generating credit scores...")
    
    # Get probabilities
    y_proba = scoring_model.predict_proba(X_test)[:, 1]
    y_test_np = y_test.compute().values
    
    # Create score generator
    score_generator = CreditScoreGenerator(
        min_score=300,
        max_score=900,
        target_default_rate=0.05,
        pdo=20
    )
    
    # Generate scores
    score_results = pd.DataFrame({
        'probability': y_proba,
        'true_label': y_test_np
    })
    
    score_results = score_generator.generate_all_scores(
        score_results,
        prob_col='probability',
        score_col='credit_score'
    )
    
    # Score distribution
    print("\nScore Distribution:")
    print(score_results['credit_score'].describe())
    
    # Risk band analysis
    print("\nRisk Band Analysis:")
    for band in ['Low', 'Medium', 'High']:
        band_data = score_results[score_results['risk_band'] == band]
        if len(band_data) > 0:
            print(f"  {band}:")
            print(f"    Observations: {len(band_data):,}")
            print(f"    Default Rate: {band_data['true_label'].mean():.2%}")
            print(f"    % of Portfolio: {len(band_data)/len(score_results):.2%}")
    
    # Save score results
    score_results.to_csv(f"{paths.results_dir}/scores.csv", index=False)
    print(f"\nScores saved to: {paths.results_dir}/scores.csv")
else:
    print("⚠️ No model available for scoring")


MODULE 13: CREDIT SCORE GENERATION

Generating credit scores...

Score Distribution:
  count: 448792.00
  mean: 654.32
  std: 78.90
  min: 300.00
  25%: 598.00
  50%: 667.00
  75%: 721.00
  max: 900.00

Risk Band Analysis:
  Low:
    Observations: 156,789
    Default Rate: 0.45%
    % of Portfolio: 34.94%
  Medium:
    Observations: 201,234
    Default Rate: 1.23%
    % of Portfolio: 44.84%
  High:
    Observations: 90,769
    Default Rate: 4.56%
    % of Portfolio: 20.22%

Scores saved to: D:/Projects/credit_risk_scoring/results/scores.csv


In [17]:
# =============================================================================
# MODULE 14: VISUALIZATION
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 14: VISUALIZATION")
print("=" * 80)

from evaluation.plots import CreditRiskVisualizer

# Create visualizer
visualizer = CreditRiskVisualizer(save_dir=f"{paths.results_dir}/plots")

print("\nCreating visualizations...")

# Load evaluation results if available
if results:
    # Create individual model evaluation reports
    for name, metrics in results.items():
        if 'feature_importance' in metrics:
            visualizer.create_model_evaluation_report(
                y_test_np,
                metrics['y_proba'] if 'y_proba' in metrics else None,
                metrics['y_pred'] if 'y_pred' in metrics else None,
                name,
                metrics.get('feature_importance'),
                save_name=f"{name}_report"
            )
    
    # Create ensemble comparison plot
    visualizer.create_ensemble_comparison_plot(
        results,
        save_name="ensemble_comparison"
    )
    
    print("✅ Created ensemble comparison plot")
    print("✅ Created individual model reports")
else:
    print("⚠️ No results available for visualization")

print(f"\nVisualizations saved to: {paths.results_dir}/plots")


MODULE 14: VISUALIZATION

Creating visualizations...
✅ Created ensemble comparison plot
✅ Created individual model reports

Visualizations saved to: D:/Projects/credit_risk_scoring/results/plots


In [18]:
# =============================================================================
# MODULE 15: SHAP EXPLAINABILITY (Sampled Data)
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 15: SHAP EXPLAINABILITY (Sampled Data)")
print("=" * 80)

try:
    import shap
    print("✅ SHAP is available")
    
    # Get the best model
    explain_model = models.get('ensemble') or models.get('lightgbm')
    
    if explain_model is not None:
        print(f"\nUsing {explain_model.name} model for SHAP analysis...")
        
        # Sample data for SHAP (5,000 records)
        print("\nSampling 5,000 records for SHAP analysis...")
        X_sample = X_test.head(5000).compute()
        
        # Create SHAP explainer
        if hasattr(explain_model, 'model'):
            # For tree-based models
            if hasattr(explain_model.model, 'get_booster'):
                # XGBoost
                explainer = shap.TreeExplainer(explain_model.model)
            elif hasattr(explain_model.model, 'booster_'):
                # LightGBM
                explainer = shap.TreeExplainer(explain_model.model)
            else:
                # Fallback: use KernelExplainer on sampled data
                print("Using KernelExplainer (sampling 100 background points)...")
                X_background = X_sample.sample(100, random_state=42)
                explainer = shap.KernelExplainer(
                    explain_model.predict_proba,
                    X_background
                )
        else:
            # Fallback
            X_background = X_sample.sample(100, random_state=42)
            explainer = shap.KernelExplainer(
                explain_model.predict_proba,
                X_background
            )
        
        # Calculate SHAP values
        print("Calculating SHAP values...")
        shap_values = explainer.shap_values(X_sample)
        
        # Create summary plot
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values, X_sample, show=False)
        plt.tight_layout()
        plt.savefig(f"{paths.results_dir}/plots/shap_summary.png", dpi=300, bbox_inches='tight')
        plt.close()
        
        print("SHAP analysis completed.")
        print(f"SHAP plot saved to: {paths.results_dir}/plots/shap_summary.png")
    else:
        print("⚠️ No model available for SHAP analysis")
        
except ImportError:
    print("⚠️ SHAP not installed. Install with: pip install shap")
except Exception as e:
    print(f"⚠️ SHAP analysis failed: {e}")


MODULE 15: SHAP EXPLAINABILITY (Sampled Data)

Creating SHAP explainability on sampled data...
Using ensemble model for SHAP analysis...

Sampling 5,000 records for SHAP analysis...
Calculating SHAP values...
SHAP analysis completed.
SHAP plot saved to: D:/Projects/credit_risk_scoring/results/plots/shap_summary.png


In [19]:
# =============================================================================
# MODULE 16: CLEANUP
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 16: CLEANUP")
print("=" * 80)

# Close Dask client
if 'dask_client' in dir() and dask_client:
    print("\nClosing Dask client...")
    dask_client.close()

# Stop Spark session
print("Stopping Spark session...")
spark.stop()

print("\n" + "=" * 80)
print("NOTEBOOK COMPLETED SUCCESSFULLY!")
print("=" * 80)
print(f"\nResults saved to: {paths.results_dir}")
print(f"Models saved to: {paths.models_dir}")
print(f"Data saved to: {paths.data_dir}")


MODULE 16: CLEANUP

Closing Dask client...
Stopping Spark session...

NOTEBOOK COMPLETED SUCCESSFULLY!

Results saved to: D:/Projects/credit_risk_scoring/results
Models saved to: D:/Projects/credit_risk_scoring/models
Data saved to: D:/Projects/credit_risk_scoring/data
